# 01 — Build the universe: membership, identifiers, exposures

The universe counterpart of the sample's `run_pipeline.ipynb`. It produces the
same two tables in the same shape, so 02–08 run on them unchanged:

| table | grain | contents |
|---|---|---|
| `ryan_<profile>_panel` | `(id, eom)` | raw characteristics, identifiers, sector, membership flags |
| `ryan_<profile>_exposures` | `(id, eom)` | standardized styles `f_*`, industry dummies `ind_*`, coverage flag |

**What differs from the sample.** The sample is a fixed 2005 cohort of 220 names.
Here membership is **point-in-time**: names enter at listing or reconstitution and
leave at delisting, and every membership decision at date *t* uses only data
known at *t*. There is no survival screen (`MIN_RETURN_MONTHS` biased specific
risk low). The pull is by date range, one year per query, rather than
`id IN (...)`.

**Profiles** (`BARRA_UNIVERSE` env var, default `r1000`):

- `full`: every JKP US common stock passing the screens (nano caps excluded).
- `r1000`: estimated on a Russell 3000 proxy (top 3000 by cap, reconstituted
  each June from May ranks), with coverage = a Russell 1000 proxy (top 1000,
  ±2.5% cumulative-cap band). The broader estimation universe gives a name
  joining the index a residual history before it joins. `config.CONSTITUENTS_CSV`
  swaps in actual constituents if you have them.

⚠️ **This hits WRDS** unless the panel table already exists. Check the flags in
the run-mode cell first.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

# ryan/universe/config.py -- imported FIRST: it shadows the sample's config for
# every shared module and puts ryan/sample (where the model code lives) on the path.
import config
assert config.PROFILE != "sample", "wrong config on the path — run this notebook from ryan/universe"

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import db
import gics
import membership
import sampling
import standardize

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

INK, ACCENT, MUTED, GREEN = "#2a78d6", "#e34948", "#8a8a83", "#1baf7a"


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


print(config.describe())
print(f"descriptors         {', '.join(config.all_descriptors())}")
print(f"dependent variable  {config.RETURN_COL}  (never standardized)")

In [ ]:
engine = db.get_engine()
db.check_connection(engine)

---
## Run mode

Same resume pattern as the sample. With the panel already stored, nothing
touches WRDS, and stage 3 (exposures) always recomputes because it is pure
pandas and the stage you iterate on.

In [ ]:
RELOAD_PANEL = False   # True → re-pull from WRDS (the whole universe: minutes, not seconds)

have_panel = db.table_exists(engine, config.TBL_PANEL)
build_panel = RELOAD_PANEL or not have_panel
print(f"{config.TBL_PANEL:<28} {'found' if have_panel else 'absent':<7} → "
      f"{'REBUILD from WRDS' if build_panel else 'resume from database'}")
print(f"{'stage 3 exposures':<28} {'':<7} → ALWAYS recomputed")


def skip(reason="already in TimescaleDB"):
    print(f"⏭  skipped — {reason}")

In [ ]:
if build_panel:
    import wrds_io
    import wrds_bulk

    conn = wrds_io.connect()
else:
    conn = None
    skip("no WRDS work required this run")

---
## Stage 1 — Pull the universe

Every JKP US row passing the JKP screens (common, main exchange, primary
security, `obs_main`), by date range. Under the R1000 profile the pull starts at
the May rank date *before* `START_EOM`, because the membership in force at
`START_EOM` was decided then.

In [ ]:
if build_panel:
    end = config.END_EOM or wrds_bulk.latest_eom(conn)
    start = wrds_bulk.pull_start()
    print(f"pulling {start} → {end}")
    raw = wrds_bulk.fetch_panel_range(conn, start, end)
else:
    skip()

In [ ]:
if build_panel:
    per = raw.groupby("eom").agg(n=("id", "size"), with_ret=(config.RETURN_COL, lambda s: s.notna().sum()))
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(per.index, per["n"], color=INK, lw=1.8)
    ax.plot(per.index, per["with_ret"], color=ACCENT, lw=1.8)
    ax.annotate(" rows", (per.index[-1], per["n"].iloc[-1]), color=INK, fontsize=9, va="center")
    ax.annotate(" with a return", (per.index[-1], per["with_ret"].iloc[-1]), color=ACCENT, fontsize=9, va="bottom")
    ax.set(title="JKP US universe per month (before membership)", ylabel="securities")
    ax.margins(x=.1)
    tidy(ax)
    plt.tight_layout()
    plt.show()
    print(f"return coverage {raw[config.RETURN_COL].notna().mean():.1%}  — the full-universe pull that")
    print("started this project had returns in 1 month of 3; anything under ~90% here is worth chasing.")
else:
    skip()

---
## Stage 2 — Identifiers and industry

The date-aware joins from the sample, unchanged: CCM links resolved per
`(id, eom)` inside each link's validity window, tickers as-of each month
(display only), GICS from `comp.company`.

GICS is the **current** classification applied back in time, which is the
sample's known look-ahead. Point-in-time GICS (`comp.co_hgic`) is deliberately
not switched on. The 2016 creation of Real Estate (sector 60, previously inside
Financials) and the 2018 expansion of Communication Services would make those
factors **nascent**: empty before their creation date. The rolling F in 03
needs every factor observed in every month of its window. See the README.

In [ ]:
if build_panel:
    links = wrds_io.fetch_ccm_links(conn)
    panel = sampling.attach_permno(raw, links)
    names = wrds_bulk.fetch_ticker_history(conn, panel["permno_ccm"].dropna().unique())
    panel = sampling.attach_ticker(panel, names)
    gics_df = wrds_io.fetch_gics(conn)
    panel = gics.attach_names(panel.merge(gics_df, on="gvkey", how="left"))
    del raw
else:
    skip()

---
## Stage 3 — Membership

Ranking uses every screened name, whether or not it has a GICS code. The real
index doesn't care about our classification, so dropping unclassified names
before ranking would shift the boundary. Names that can't be modelled (no
industry) are removed from the estimation universe *after* ranking, with a count.

In [ ]:
if build_panel:
    panel = membership.apply_profile(panel, config)
    panel = panel[panel["eom"] >= pd.Timestamp(config.START_EOM)]

    no_ind = panel["in_estu"] & panel[config.INDUSTRY_CODE_COL].isna()
    print(f"estimation-universe rows without a {config.INDUSTRY_CODE_COL}: {no_ind.sum():,} "
          f"({no_ind.sum() / max(panel['in_estu'].sum(), 1):.2%}) — removed, they cannot be modelled")
    panel.loc[no_ind, "in_estu"] = False
    if config.COVERAGE_COL:
        lost = panel[config.COVERAGE_COL] & no_ind
        print(f"   of which in coverage: {lost.sum():,}")
        panel.loc[no_ind, config.COVERAGE_COL] = False
else:
    skip()

In [ ]:
if build_panel:
    flags = ["in_estu"] + ([config.COVERAGE_COL] if config.COVERAGE_COL else [])
    counts = panel.groupby("eom")[flags].sum()
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
    for f, c in zip(flags, (INK, ACCENT)):
        ax[0].plot(counts.index, counts[f], color=c, lw=1.8)
        ax[0].annotate(f" {f}", (counts.index[-1], counts[f].iloc[-1]), color=c, fontsize=9, va="center")
    ax[0].set(title="Members per month", ylabel="names")
    ax[0].margins(x=.12)
    tv_flag = config.COVERAGE_COL or "in_estu"
    tv = membership.turnover(panel, panel[tv_flag])
    ax[1].bar(tv.index, tv["joined"], width=25, color=GREEN, label="joined")
    ax[1].bar(tv.index, -tv["left"], width=25, color=ACCENT, label="left")
    ax[1].set(title=f"Turnover of {tv_flag}", ylabel="names")
    ax[1].legend(frameon=False, fontsize=9)
    for a in ax:
        tidy(a)
    plt.tight_layout()
    plt.show()
    display(counts.describe().round(0).T)
    if config.COVERAGE_COL:
        ann_turn = tv["joined"].groupby(tv.index.year).sum() / tv["members"].groupby(tv.index.year).mean()
        print(f"annual one-way turnover of the coverage universe: median {ann_turn.median():.1%} "
              "(the real Russell 1000 runs roughly 5–10%)")
else:
    skip()

In [ ]:
if build_panel:
    panel = panel[panel["in_estu"]].reset_index(drop=True)
    checks = {
        "(id, eom) unique — no join fan-out": not panel.duplicated(["id", "eom"]).any(),
        "every row has an industry": panel[config.INDUSTRY_CODE_COL].notna().all(),
        "all 11 sectors present": panel["sector_code"].nunique() == 11,
        "first month is START_EOM": panel["eom"].min() == pd.Timestamp(config.START_EOM),
        "returns present in most rows": panel[config.RETURN_COL].notna().mean() > 0.9,
    }
    if config.COVERAGE_COL:
        checks["coverage ⊆ estimation universe"] = True   # enforced by apply_profile
    for k, ok in checks.items():
        print(f"{'✅' if ok else '❌'} {k}")
    print(f"\n{len(panel):,} rows, {panel['id'].nunique():,} securities, "
          f"{panel['eom'].nunique()} months ({panel['eom'].min().date()} → {panel['eom'].max().date()})")
    db.write_table(panel, config.TBL_PANEL, engine, unique_on=["id", "eom"])
else:
    panel = db.read_table(config.TBL_PANEL, engine)
    panel["eom"] = pd.to_datetime(panel["eom"])
    print(f"📂 loaded {len(panel):,} rows / {panel['id'].nunique():,} securities from {config.TBL_PANEL}")
    print(f"   {panel['eom'].min().date()} → {panel['eom'].max().date()}")

In [ ]:
nulls = (panel[config.all_descriptors() + [config.RETURN_COL, config.CAP_COL]].isna().mean() * 100)
nulls.sort_values(ascending=False).round(2).to_frame("% missing").T

---
## Stage 4 — Exposures

Exactly the sample's transform: robust winsorization, cap-weighted mean /
equal-weighted sd, log size, descriptor combination, one dummy per industry.
Then the **industry gate**. The sample plotted the thinnest industry and relied
on someone looking; here a month with an industry under `MIN_INDUSTRY_NAMES`
stops the build.

In [ ]:
exposures = standardize.standardize_panel(panel)
print("\ncombining collinear descriptors into one factor each:")
exposures = standardize.combine_descriptors(exposures)
exposures, dummy_cols = standardize.build_industry_dummies(exposures, code_col=config.INDUSTRY_CODE_COL)
factor_cols = [c for c in exposures.columns if c.startswith("f_")]
print(f"\nK = {1 + len(dummy_cols) + len(factor_cols)}: market + {len(dummy_cols)} industries + {len(factor_cols)} styles"
      f"  →  T/K on a {config.COV_WINDOW}-month window = {config.COV_WINDOW / (1 + len(dummy_cols) + len(factor_cols)):.1f}")

In [ ]:
ind_counts = membership.industry_gate(exposures, dummy_cols, config.MIN_INDUSTRY_NAMES)
fig, ax = plt.subplots(figsize=(13, 3))
thin = ind_counts.min(axis=1)
ax.plot(thin.index, thin, color=INK, lw=1.8)
ax.axhline(config.MIN_INDUSTRY_NAMES, color=ACCENT, lw=1, ls="--")
ax.annotate(f" gate: {config.MIN_INDUSTRY_NAMES}", (thin.index[0], config.MIN_INDUSTRY_NAMES), color=ACCENT,
            fontsize=9, va="bottom")
ax.set(title="Thinnest industry per month", ylabel="names", ylim=(0, None))
tidy(ax)
plt.tight_layout()
plt.show()
print(f"✅ every industry has ≥ {config.MIN_INDUSTRY_NAMES} names in every month "
      f"(minimum {int(thin.min())}, in {thin.idxmin().date()})")

In [ ]:
w = pd.to_numeric(exposures[config.CAP_COL], errors="coerce")
rows = []
for col in factor_cols:
    x, g = exposures[col], exposures["eom"]
    ok = x.notna() & w.notna() & (w > 0)
    cwm = (pd.DataFrame({"xw": (x * w).where(ok), "w": w.where(ok), "g": g})
           .groupby("g").sum(min_count=1).pipe(lambda d: d["xw"] / d["w"]))
    rows.append({"factor": col, "max |cap-wtd mean|": cwm.abs().max(),
                 "max |sd - 1|": (x.groupby(g).std() - 1).abs().max(),
                 "max |z|": x.abs().max(), "% missing": x.isna().mean() * 100})
summary = pd.DataFrame(rows).set_index("factor")
display(summary.round(4))
print(f"{'✅' if summary['max |cap-wtd mean|'].max() < 1e-8 else '❌'} cap-weighted mean is zero in every cross-section")
print(f"{'✅' if summary['max |sd - 1|'].max() < 1e-8 else '❌'} unit dispersion in every cross-section")
print(f"{'✅' if summary['max |z|'].max() < 6 else '❌'} no runaway exposures (max |z| = {summary['max |z|'].max():.2f})")

In [ ]:
# The value-descriptor decision (handover #7) was made on the sample: be_me alone
# gave value ≈ −0.33 against both momentum and profitability. Check it holds on a
# universe 15–20× larger before trusting the factor's name.
corr = exposures[factor_cols].corr()
names_ = [c[2:] for c in factor_cols]
fig, ax = plt.subplots(figsize=(6.5, 5.2))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(names_)), names_, rotation=45, ha="right")
ax.set_yticks(range(len(names_)), names_)
for i in range(len(names_)):
    for j in range(len(names_)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > .55 else "#1a1a19")
plt.colorbar(im, shrink=.8)
ax.set_title("Style exposure correlations (pooled)")
plt.tight_layout()
plt.show()
print(f"value↔momentum {corr.loc['f_value', 'f_momentum']:+.3f}, value↔profitability "
      f"{corr.loc['f_value', 'f_profitability']:+.3f}   (sample: −0.33 / −0.33)")

In [ ]:
keep = (["id", "eom", "gvkey", "permno_ccm", "ticker", "conm", "size_grp",
         "sector_code", "sector_name", "group_code", "group_name",
         config.RETURN_COL, config.CAP_COL]
        + ([config.COVERAGE_COL] if config.COVERAGE_COL else [])
        + factor_cols + dummy_cols)
out = exposures[[c for c in keep if c in exposures.columns]].copy()
db.write_table(out, config.TBL_EXPOSURES, engine, unique_on=["id", "eom"])
print(f"{len(out):,} rows × {out.shape[1]} columns")

---
## Next

02 → 03 → 04 → 05, then 06–08. Same notebooks as the sample, reading this
profile's tables. Things to check on the first pass, in order:

1. **02**: market factor vs the cap-weighted universe return (sample: 0.998);
   style vols 2–8%/yr; median R² (lower than the sample's 0.27 is expected with
   more small names).
2. **03**: the NW table should still favour zero lags; the eigen adjustment should
   still be doing the heavy lifting.
3. **05**: min-variance bias, raw vs final. That is the number that says whether
   07 can be trusted at this scale.
4. Record 03's printed factor means/vols in `config.ANCHORS` once you trust them.